# pctrans — Colab Quickstart

**Pre-Clinical to Clinical Translation:** a dual-tower contrastive model that aligns CCLE cancer cell lines to TCGA patients in a shared 64-dimensional manifold.

This notebook runs the whole pipeline end-to-end on a free Colab T4 (or CPU): download → preprocess → train → evaluate → UMAP. Runtime is dominated by the one-time TCGA download (~1.5 GB); training itself is ~3 min on T4 / ~12 min on CPU.

- Repo: https://github.com/vthawfeek/pre-clinical-to-clinical-translation
- Live demo: https://pctrans.streamlit.app

> Phase 1 result (3 lineages, held-out test set): kNN@5 = 100%, silhouette +0.57, TFS 0.89. Phase 2 stress-tests this with confidence intervals, real batch-correction baselines, a 15-lineage task, a purity confounder analysis, a label-shuffle control, and a vemurafenib/BRAF case study.

## 1. Install

Installs `pctrans` and its dependencies straight from GitHub. On Colab, torch is preinstalled, so this is fast.

In [ ]:
!pip install -q "git+https://github.com/vthawfeek/pre-clinical-to-clinical-translation.git"
import pctrans
print("pctrans", pctrans.__version__)

## 2. Download data

CCLE RNA-seq (DepMap 24Q4) and TCGA RNA-seq (UCSC Xena Pan-Cancer). Both are public and unrestricted — no access application needed. The TCGA expression matrix is the large one (~1.5 GB uncompressed); this cell is the slow step.

In [ ]:
!pctrans-download ccle --out-dir data/raw/ccle/
!pctrans-download tcga --out-dir data/raw/tcga/

## 3. Preprocess

Harmonise gene IDs (strip Entrez suffixes), take the intersection of CCLE/TCGA genes, select the top 2,000 highly variable genes by the union-rank method, then make a lineage-stratified train/val/test split with train-only z-score scalers.

In [ ]:
!pctrans-preprocess --raw-dir data/raw/ --out-dir data/processed/ --n-hvgs 2000 --split

## 4. Train

30-epoch SupCon-InfoNCE run with a learnable temperature, cosine LR schedule, and early stopping on validation kNN@5. Best checkpoint is saved to `models/best_model.pt`.

In [ ]:
!pctrans-train --config configs/training.yaml --data-dir data/processed/

## 5. Evaluate (Gate 1)

Cross-domain kNN@{1,3,5,10} retrieval accuracy on the held-out test set, cross-domain silhouette, and the composite Translational Fidelity Score.

In [ ]:
!pctrans-evaluate --model models/best_model.pt --data-dir data/processed/

## 6. UMAP reveal

Project the aligned test embeddings to 2-D and render the lineage/domain scatter. If the model worked, each lineage forms one cluster containing *both* CCLE cell lines (✕) and TCGA patients (●).

In [ ]:
!pctrans-visualize --model models/best_model.pt --data-dir data/processed/ --out-dir reports/

from IPython.display import Image, display
display(Image("reports/umap_test_set.png"))

## Next steps

- Explore any cell line interactively in the live demo: https://pctrans.streamlit.app
- Read the docs in `docs/` for the data pipeline, architecture, training, and evaluation details.
- Phase 2 (`PLAN-phase2.md`) hardens these numbers: CIs, real baselines (Harmony/ComBat/Scanorama), a 15-lineage task, a tumour-purity confounder check, a label-shuffle negative control, and a vemurafenib/BRAF case study.